# MILK10k — Dual-View DINOv2 ViT + Latent Cross-Attention Fusion (Single GPU)

This notebook is built for the **MILK10k** folder layout you showed:
- `images/`
- `licenses/`
- `supplements/`
- `attribution` (txt)
- `metadata.xlsx` (or `metadata.csv`)

It uses the dataset's **hierarchical diagnosis fields** (`diagnosis_1..4`, `diagnosis_2`) to map each lesion to the **11 benchmark categories**:
`AKIEC, BCC, BEN_OTH, BKL, DF, INF, MAL_OTH, MEL, NV, SCCKA, VASC`.

**Metric**: Macro-F1 after binarizing each class at **≥ 0.5**.

> Notes:
> - If you later find MONET concept scores / skin tone columns inside `supplements/`, you can merge them (scaffold included).
> - Designed to run on a single **4GB GPU**: AMP + gradient accumulation + staged unfreezing.


## 0) Install dependencies (run once)

In [ ]:
!pip -q install pandas numpy scikit-learn pillow tqdm torchvision openpyxl

## 1) Imports + device + reproducibility

In [1]:

import os, math, random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 2) Config (EDIT paths if needed)

In [ ]:

@dataclass
class CFG:
    # Root folder that contains: images/, supplements/, metadata.xlsx, attribution, etc.
    data_root: str = "data/MILK10k"

    # Metadata file (can be .xlsx or .csv)
    meta_file: str = "metadata.xlsx"   # or "metadata.csv"

    # Optional: extra metadata file (e.g. from supplements/) containing skin tone and/or MONET concepts
    extra_meta_file: Optional[str] = None

    # Image folder + extension
    images_dir: str = "images"
    img_ext: str = ".jpg"  # change to .png if needed

    # Column names (as in MILK10k metadata)
    col_isic: str = "isic_id"
    col_lesion: str = "lesion_id"
    col_image_type: str = "image_type"

    col_age: str = "age_approx"
    col_sex: str = "sex"
    col_site_general: str = "anatom_site_general"
    col_site_special: str = "anatom_site_special"

    # Diagnosis hierarchy columns
    col_d1: str = "diagnosis_1"
    col_d2: str = "diagnosis_2"
    col_d3: str = "diagnosis_3"
    col_d4: str = "diagnosis_4"

    # Image types to pair
    image_type_clin: str = "clinical: close-up"
    image_type_derm: str = "dermoscopic"

    # 11 benchmark categories
    classes: Tuple[str, ...] = ("AKIEC","BCC","BEN_OTH","BKL","DF","INF","MAL_OTH","MEL","NV","SCCKA","VASC")

    # Training
    img_size: int = 224
    num_workers: int = 2

    micro_batch: int = 4
    grad_accum: int = 8

    epochs_stage1: int = 12
    epochs_stage2: int = 30

    lr_heads: float = 3e-4
    lr_lora: float = 1e-4
    lr_unfreeze: float = 2e-5
    weight_decay: float = 0.05

    # LoRA
    lora_rank: int = 8
    lora_alpha: float = 16.0
    lora_dropout: float = 0.05

    # Fusion
    d_model: int = 384  # DINOv2 ViT-S embedding dim
    n_latents: int = 8
    fusion_layers: int = 3
    n_heads: int = 6

    # Loss weights
    w_ce: float = 0.5
    w_excl: float = 0.1
    w_monet: float = 0.2
    w_contrast: float = 0.05  # set 0.0 to disable

    max_grad_norm: float = 1.0
    use_amp: bool = True

cfg = CFG()
print(cfg)


## 3) Transforms

In [ ]:

train_tfms = T.Compose([
    T.Resize((cfg.img_size, cfg.img_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2, 0.05)], p=0.5),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.2),
    T.ToTensor(),
    T.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
])

val_tfms = T.Compose([
    T.Resize((cfg.img_size, cfg.img_size)),
    T.ToTensor(),
    T.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
])


## 4) Load metadata (.xlsx or .csv) + sanity checks

In [ ]:

def read_table(path: str) -> pd.DataFrame:
    p = path if os.path.isabs(path) else os.path.join(cfg.data_root, path)
    ext = os.path.splitext(p)[1].lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(p)
    return pd.read_csv(p)

meta = read_table(cfg.meta_file)
print("Metadata shape:", meta.shape)
display(meta.head())

required = [cfg.col_isic, cfg.col_lesion, cfg.col_image_type, cfg.col_age, cfg.col_sex,
            cfg.col_site_general, cfg.col_site_special, cfg.col_d1, cfg.col_d2, cfg.col_d3, cfg.col_d4]
missing = [c for c in required if c not in meta.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("image_type counts:")
print(meta[cfg.col_image_type].value_counts())
print("Unique lesions:", meta[cfg.col_lesion].nunique())


## 5) Pair clinical+derm images per lesion (one row = one lesion)
Each lesion should have exactly **2 images** (clinical + dermoscopic). We'll pivot to get both `isic_id`s and build file paths.

In [ ]:

# Pivot to get both isic_ids per lesion_id
pairs = meta.pivot_table(index=cfg.col_lesion, columns=cfg.col_image_type, values=cfg.col_isic, aggfunc='first')
pairs = pairs.rename(columns={cfg.image_type_clin: "clin_isic_id", cfg.image_type_derm: "derm_isic_id"}).reset_index()

if "clin_isic_id" not in pairs.columns or "derm_isic_id" not in pairs.columns:
    raise ValueError("Couldn't find both clinical and derm views. Check cfg.image_type_clin / cfg.image_type_derm.")

# Lesion-level metadata (diagnosis + demographics) from first row per lesion
les = meta.groupby(cfg.col_lesion).first().reset_index()

lesions = pairs.merge(
    les[[cfg.col_lesion, cfg.col_age, cfg.col_sex, cfg.col_site_general, cfg.col_site_special,
         cfg.col_d1, cfg.col_d2, cfg.col_d3, cfg.col_d4]],
    on=cfg.col_lesion, how="left"
)

def make_img_path(isic_id: str) -> str:
    return os.path.join(cfg.data_root, cfg.images_dir, f"{isic_id}{cfg.img_ext}")

lesions["clin_path"] = lesions["clin_isic_id"].apply(make_img_path)
lesions["derm_path"] = lesions["derm_isic_id"].apply(make_img_path)

# Site
def resolve_site(row):
    g = row[cfg.col_site_general]
    s = row[cfg.col_site_special]
    if pd.notna(g) and str(g).strip() != "":
        return str(g)
    if pd.notna(s) and str(s).strip() != "":
        return str(s)
    return "unknown"

lesions["anatom_site"] = lesions.apply(resolve_site, axis=1)

# Age 5y bin
def age_to_bin(a):
    if pd.isna(a):
        return "unknown"
    try:
        a = float(a)
        b = int(5 * round(a / 5.0))
        return str(b)
    except:
        return "unknown"

lesions["age_bin_5y"] = lesions[cfg.col_age].apply(age_to_bin)
lesions["sex_norm"] = lesions[cfg.col_sex].fillna("unknown").astype(str).str.lower()

print("Lesion-level table:", lesions.shape)
display(lesions.head())


## 6) Map hierarchical diagnoses → 11 benchmark categories
MILK10k diagnoses are mapped to a simplified ISIC/HAM-style 11-category scheme.
This mapping is **rule-based** using `diagnosis_2..4` (and fallback rules for collision / missing diagnosis_3).

In [ ]:

CLASSES = list(cfg.classes)
CLASS2IDX = {c:i for i,c in enumerate(CLASSES)}

# Primary mapping based on diagnosis_3/4 (most lesions)
MAP_D3 = {
    # Malignant
    "Basal cell carcinoma": "BCC",
    "Squamous cell carcinoma in situ": "SCCKA",
    "Squamous cell carcinoma, Invasive": "SCCKA",
    "Keratoacanthoma": "SCCKA",
    "Melanoma Invasive": "MEL",
    "Melanoma in situ": "MEL",
    "Melanoma metastasis": "MEL",

    # Pre-malignant / indeterminate
    "Solar or actinic keratosis": "AKIEC",

    # Melanocytic benign
    "Nevus": "NV",

    # DF
    "Dermatofibroma": "DF",

    # Vascular
    "Hemangioma": "VASC",
    "Pyogenic granuloma": "VASC",
    "Angiokeratoma": "VASC",

    # Benign keratinocytic lesions (BKL)
    "Seborrheic keratosis": "BKL",
    "Lichen planus like keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Porokeratosis": "BKL",
    "Clear cell acanthoma": "BKL",
    "Ink-spot lentigo": "BKL",

    # Inflammatory / infectious
    "Molluscum": "INF",
    "Mastocytosis": "INF",

    # Other benign
    "Infundibular or epidermal cyst": "BEN_OTH",
    "Sebaceous hyperplasia": "BEN_OTH",
    "Supernumerary nipple": "BEN_OTH",
    "Juvenile xanthogranuloma": "BEN_OTH",
    "Mucosal melanotic macule": "BEN_OTH",
    "Trichoblastoma": "BEN_OTH",
}

def map_row_to_category(row) -> str:
    d1 = str(row.get(cfg.col_d1, "")).strip()
    d2 = str(row.get(cfg.col_d2, "")).strip()
    d3 = row.get(cfg.col_d3, None)
    d4 = row.get(cfg.col_d4, None)

    d3s = str(d3).strip() if pd.notna(d3) else ""
    d4s = str(d4).strip() if pd.notna(d4) else ""

    # Diagnosis_4 sometimes adds extra specificity
    if d4s == "Squamous cell carcinoma in situ, Bowens disease":
        return "SCCKA"
    if d4s == "Blue nevus" or d4s.startswith("Nevus,"):
        return "NV"
    if "Hemangioma" in d4s:
        return "VASC"

    # Primary mapping by diagnosis_3
    if d3s in MAP_D3:
        return MAP_D3[d3s]

    # Fallbacks when diagnosis_3 missing or unrecognized
    d2_low = d2.lower()

    # Collisions
    if "collision" in d2_low:
        if "malignant" in d2_low:
            return "MAL_OTH"
        return "BEN_OTH"

    # Explicit inflammatory bucket (many rows have diagnosis_3 missing here)
    if "inflammatory" in d2_low or "infectious" in d2_low:
        return "INF"

    # Exogenous (e.g., ink/foreign material) -> other benign
    if "exogenous" in d2_low:
        return "BEN_OTH"

    # Benign vascular bucket
    if "vascular" in d2_low:
        return "VASC"

    # If still unknown:
    if d1.lower() == "malignant":
        return "MAL_OTH"
    return "BEN_OTH"

lesions["target"] = lesions.apply(map_row_to_category, axis=1)

# Report
print("Target distribution:")
display(lesions["target"].value_counts())

unknown = lesions[~lesions["target"].isin(CLASSES)]
print("Unknown targets:", unknown.shape[0])
if len(unknown) > 0:
    display(unknown[[cfg.col_lesion, cfg.col_d1, cfg.col_d2, cfg.col_d3, cfg.col_d4, "target"]].head(30))

lesions["y_idx"] = lesions["target"].map(CLASS2IDX).astype(int)


## 7) Optional: merge extra metadata (skin tone / MONET) from supplements
If you find extra metadata in `supplements/` (e.g. skin tone or MONET concept probabilities), set `cfg.extra_meta_file`.
This notebook will detect `skin_tone` and any columns containing `monet`.

In [ ]:

monet_cols = []
lesions["skin_tone"] = "unknown"

if cfg.extra_meta_file is None:
    print("No extra_meta_file provided. Proceeding without skin tone / MONET.")
else:
    extra = read_table(cfg.extra_meta_file)
    print("Extra meta shape:", extra.shape)
    display(extra.head())

    # Find lesion key
    key = None
    for c in ["lesion", "lesion_id", cfg.col_lesion]:
        if c in extra.columns:
            key = c
            break
    if key is None:
        raise ValueError("Could not find lesion id column in extra_meta_file.")
    extra = extra.rename(columns={key: cfg.col_lesion})

    # skin tone
    tone_col = None
    for c in ["skin_tone", "skinTone", "tone", "skin_tone_category"]:
        if c in extra.columns:
            tone_col = c
            break
    if tone_col is not None:
        extra["skin_tone"] = extra[tone_col].fillna("unknown").astype(str)
        lesions = lesions.merge(extra[[cfg.col_lesion, "skin_tone"]], on=cfg.col_lesion, how="left")

    # MONET columns
    monet_cols = [c for c in extra.columns if "monet" in str(c).lower()]
    if monet_cols:
        keep = [cfg.col_lesion] + monet_cols
        lesions = lesions.merge(extra[keep], on=cfg.col_lesion, how="left")
        print("Found MONET cols:", len(monet_cols))
    else:
        print("No MONET columns detected.")

display(lesions.head())


## 8) Build vocabularies for categorical metadata

In [ ]:

class Vocab:
    def __init__(self):
        self.stoi = {"<UNK>": 0}
        self.itos = ["<UNK>"]

    def build(self, values: List[str]):
        for v in values:
            v = str(v)
            if v not in self.stoi:
                self.stoi[v] = len(self.itos)
                self.itos.append(v)

    def encode(self, v) -> int:
        return self.stoi.get(str(v), 0)

    def __len__(self):
        return len(self.itos)

v_age  = Vocab(); v_age.build(lesions["age_bin_5y"].tolist())
v_sex  = Vocab(); v_sex.build(lesions["sex_norm"].tolist())
v_site = Vocab(); v_site.build(lesions["anatom_site"].tolist())
v_tone = Vocab(); v_tone.build(lesions["skin_tone"].fillna("unknown").astype(str).tolist())

print("Vocab sizes:", len(v_age), len(v_sex), len(v_site), len(v_tone))


## 9) Train/val split (lesion-level stratified)

In [ ]:

from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
idx_train, idx_val = next(splitter.split(lesions[cfg.col_lesion], lesions["y_idx"]))
df_train = lesions.iloc[idx_train].reset_index(drop=True)
df_val   = lesions.iloc[idx_val].reset_index(drop=True)

print("Train lesions:", len(df_train), "Val lesions:", len(df_val))
print("Train label counts:")
display(df_train["target"].value_counts())


## 10) Dataset (one item = one lesion with two images)

In [ ]:

class MILK10kLesionPairs(Dataset):
    def __init__(self, df: pd.DataFrame, tfms, monet_cols: List[str]):
        self.df = df
        self.tfms = tfms
        self.monet_cols = monet_cols

    def _load_img(self, p: str) -> torch.Tensor:
        img = Image.open(p).convert("RGB")
        return self.tfms(img)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i: int):
        row = self.df.iloc[i]
        img_clin = self._load_img(row["clin_path"])
        img_derm = self._load_img(row["derm_path"])

        cat = {
            "age":  torch.tensor(v_age.encode(row["age_bin_5y"]), dtype=torch.long),
            "sex":  torch.tensor(v_sex.encode(row["sex_norm"]), dtype=torch.long),
            "site": torch.tensor(v_site.encode(row["anatom_site"]), dtype=torch.long),
            "tone": torch.tensor(v_tone.encode(row["skin_tone"]), dtype=torch.long),
        }

        if len(self.monet_cols) > 0:
            m = row[self.monet_cols].values.astype(np.float32)
            m = np.nan_to_num(m, nan=0.0)
            monet = torch.tensor(m, dtype=torch.float32)
        else:
            monet = torch.zeros((0,), dtype=torch.float32)

        y_idx = torch.tensor(int(row["y_idx"]), dtype=torch.long)
        y_oh = torch.zeros((len(cfg.classes),), dtype=torch.float32)
        y_oh[int(row["y_idx"])] = 1.0

        return {
            "lesion": str(row[cfg.col_lesion]),
            "img_clin": img_clin,
            "img_derm": img_derm,
            "cat": cat,
            "monet": monet,
            "y_idx": y_idx,
            "y_oh": y_oh
        }

def collate_fn(batch):
    out = {}
    out["lesion"] = [b["lesion"] for b in batch]
    out["img_clin"] = torch.stack([b["img_clin"] for b in batch], dim=0)
    out["img_derm"] = torch.stack([b["img_derm"] for b in batch], dim=0)
    out["y_idx"] = torch.stack([b["y_idx"] for b in batch], dim=0)
    out["y_oh"] = torch.stack([b["y_oh"] for b in batch], dim=0)
    out["cat"] = {k: torch.stack([b["cat"][k] for b in batch], dim=0) for k in batch[0]["cat"].keys()}
    if batch[0]["monet"].numel() > 0:
        out["monet"] = torch.stack([b["monet"] for b in batch], dim=0)
    else:
        out["monet"] = torch.zeros((len(batch), 0), dtype=torch.float32)
    return out

train_ds = MILK10kLesionPairs(df_train, train_tfms, monet_cols)
val_ds   = MILK10kLesionPairs(df_val, val_tfms, monet_cols)

print("Example:", train_ds[0]["lesion"], train_ds[0]["img_clin"].shape, train_ds[0]["monet"].shape, train_ds[0]["y_idx"])


## 11) DataLoaders + imbalance handling

In [ ]:

# Weighted sampling for macro-F1 friendliness
class_counts = df_train["y_idx"].value_counts().to_dict()
weights = df_train["y_idx"].map(lambda c: 1.0 / class_counts[int(c)]).values
sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.micro_batch,
    sampler=sampler,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.micro_batch,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=False,
    collate_fn=collate_fn,
)

print("Train steps/epoch:", len(train_loader), "Val steps:", len(val_loader))


## 12) Dual-modality LoRA + patching utilities

In [ ]:

class DualLoRALinear(nn.Module):
    """Frozen Linear + two LoRA branches (modality 0=clinical, 1=derm)."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)

        self.in_features = base.in_features
        self.out_features = base.out_features
        self.r = r
        self.scaling = alpha / max(1, r)
        self.drop = nn.Dropout(dropout)

        self.A = nn.Parameter(torch.zeros(2, r, self.in_features))
        self.B = nn.Parameter(torch.zeros(2, self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

        self._modality = 0

    def set_modality(self, modality: int):
        self._modality = int(modality)

    def forward(self, x):
        base_out = self.base(x)
        A = self.A[self._modality]
        B = self.B[self._modality]
        lora = (self.drop(x) @ A.t()) @ B.t()
        return base_out + self.scaling * lora

def replace_linear_with_dual_lora(module: nn.Module, target_substrings: List[str],
                                  r: int, alpha: float, dropout: float) -> int:
    replaced = 0
    for name, child in list(module.named_children()):
        replaced += replace_linear_with_dual_lora(child, target_substrings, r, alpha, dropout)
        if isinstance(child, nn.Linear) and any(ts in name for ts in target_substrings):
            setattr(module, name, DualLoRALinear(child, r=r, alpha=alpha, dropout=dropout))
            replaced += 1
    return replaced

def set_modality_all(module: nn.Module, modality: int):
    for m in module.modules():
        if isinstance(m, DualLoRALinear):
            m.set_modality(modality)


## 13) DINOv2 token backbone

In [ ]:

class DinoV2TokenBackbone(nn.Module):
    """Loads DINOv2 from torch.hub and returns tokens: [CLS] + [REG?] + [PATCH]."""
    def __init__(self, model_name="dinov2_vits14", use_reg_tokens=True):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", model_name)
        self.use_reg_tokens = use_reg_tokens

    def forward_tokens(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone.forward_features(x)
        cls = feats["x_norm_clstoken"]              # [B, D]
        patch = feats["x_norm_patchtokens"]         # [B, 256, D] for 224/14
        reg = feats.get("x_norm_regtokens", None)   # [B, R, D] or None
        if (reg is None) or (not self.use_reg_tokens):
            reg = torch.empty((x.size(0), 0, cls.size(-1)), device=x.device, dtype=cls.dtype)
        tokens = torch.cat([cls.unsqueeze(1), reg, patch], dim=1)
        return tokens

    def forward_view(self, x: torch.Tensor, modality: int) -> torch.Tensor:
        set_modality_all(self, modality)
        return self.forward_tokens(x)


## 14) Metadata tokenizer + latent cross-attention fusion

In [ ]:

class MetaTokenizer(nn.Module):
    def __init__(self, d_model: int,
                 n_age: int, n_sex: int, n_site: int, n_tone: int,
                 monet_dim: int):
        super().__init__()
        self.age = nn.Embedding(n_age, d_model)
        self.sex = nn.Embedding(n_sex, d_model)
        self.site = nn.Embedding(n_site, d_model)
        self.tone = nn.Embedding(n_tone, d_model)

        self.monet_dim = monet_dim
        if monet_dim > 0:
            self.monet_mlp = nn.Sequential(
                nn.Linear(monet_dim, d_model * 2),
                nn.GELU(),
                nn.Linear(d_model * 2, d_model * 2),
            )
        else:
            self.monet_mlp = None

    def forward(self, cat: Dict[str, torch.Tensor], monet: torch.Tensor) -> torch.Tensor:
        toks = [
            self.age(cat["age"]),
            self.sex(cat["sex"]),
            self.site(cat["site"]),
            self.tone(cat["tone"]),
        ]
        if self.monet_dim > 0:
            mm = self.monet_mlp(monet)
            t1, t2 = mm.chunk(2, dim=-1)
            toks.extend([t1, t2])
        return torch.stack(toks, dim=1)  # [B, M, D]

class CrossAttnBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.cross = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ln_sa = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ln_mlp = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, z: torch.Tensor, kv: torch.Tensor) -> torch.Tensor:
        z2 = self.ln_q(z)
        kv2 = self.ln_kv(kv)
        z_attn, _ = self.cross(z2, kv2, kv2, need_weights=False)
        z = z + z_attn

        z2 = self.ln_sa(z)
        z_sa, _ = self.self_attn(z2, z2, z2, need_weights=False)
        z = z + z_sa

        z = z + self.mlp(self.ln_mlp(z))
        return z

class LatentFuser(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_latents: int, n_layers: int):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(n_latents, d_model) * 0.02)
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_out = nn.LayerNorm(d_model)

    def forward(self, kv: torch.Tensor) -> torch.Tensor:
        B = kv.size(0)
        z = self.latents.unsqueeze(0).expand(B, -1, -1)
        for blk in self.blocks:
            z = blk(z, kv)
        z = self.ln_out(z)
        return z.mean(dim=1)


## 15) Full model (dual-view LoRA + fusion + heads)

In [ ]:

class MILK10kViTModel(nn.Module):
    def __init__(self, monet_dim: int):
        super().__init__()
        self.backbone = DinoV2TokenBackbone(model_name="dinov2_vits14", use_reg_tokens=True)

        replaced = replace_linear_with_dual_lora(
            self.backbone,
            target_substrings=["qkv", "proj"],  # if replaced==0, inspect module names and adjust
            r=cfg.lora_rank,
            alpha=cfg.lora_alpha,
            dropout=cfg.lora_dropout
        )
        print(f"LoRA replaced {replaced} Linear layers.")

        self.meta = MetaTokenizer(cfg.d_model, len(v_age), len(v_sex), len(v_site), len(v_tone), monet_dim)
        self.fuser = LatentFuser(cfg.d_model, cfg.n_heads, cfg.n_latents, cfg.fusion_layers)

        self.head_sig = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(cfg.d_model * 2, len(cfg.classes)),
        )
        self.head_sm = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, len(cfg.classes)),
        )

        self.head_monet = None
        if monet_dim > 0:
            self.head_monet = nn.Sequential(
                nn.LayerNorm(cfg.d_model),
                nn.Linear(cfg.d_model, cfg.d_model),
                nn.GELU(),
                nn.Linear(cfg.d_model, monet_dim),
            )

        self.proj = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, 128))

    def forward(self, batch: Dict[str, torch.Tensor]):
        tok_c = self.backbone.forward_view(batch["img_clin"], modality=0)
        tok_d = self.backbone.forward_view(batch["img_derm"], modality=1)
        t_meta = self.meta(batch["cat"], batch["monet"])

        kv = torch.cat([tok_c, tok_d, t_meta], dim=1)
        z = self.fuser(kv)

        logits_sig = self.head_sig(z)
        logits_sm  = self.head_sm(z)

        monet_hat = self.head_monet(z) if self.head_monet is not None else None

        pc = F.normalize(self.proj(tok_c[:, 0]), dim=-1)
        pd = F.normalize(self.proj(tok_d[:, 0]), dim=-1)

        return {"logits_sig": logits_sig, "logits_sm": logits_sm, "monet_hat": monet_hat, "pc": pc, "pd": pd}

monet_dim = int(train_ds[0]["monet"].numel())
model = MILK10kViTModel(monet_dim).to(device)


## 16) Losses + Macro-F1@0.5 metric

In [ ]:

def info_nce(pc: torch.Tensor, pd: torch.Tensor, temperature=0.07):
    logits = (pc @ pd.t()) / temperature
    labels = torch.arange(pc.size(0), device=pc.device)
    return F.cross_entropy(logits, labels)

def macro_f1_threshold05(probs: np.ndarray, y_true_oh: np.ndarray) -> float:
    pred = (probs >= 0.5).astype(np.int32)
    true = (y_true_oh >= 0.5).astype(np.int32)
    f1s = []
    for c in range(true.shape[1]):
        tp = np.sum((pred[:,c] == 1) & (true[:,c] == 1))
        fp = np.sum((pred[:,c] == 1) & (true[:,c] == 0))
        fn = np.sum((pred[:,c] == 0) & (true[:,c] == 1))
        denom = (2*tp + fp + fn)
        f1s.append((2*tp/denom) if denom > 0 else 0.0)
    return float(np.mean(f1s))

# pos_weight for BCE
counts = df_train["y_idx"].value_counts().reindex(range(len(cfg.classes))).fillna(0).values
counts = np.maximum(counts, 1)
pos_weight = (len(df_train) - counts) / counts
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=device)

def compute_losses(out: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]):
    logits_sig = out["logits_sig"]
    logits_sm  = out["logits_sm"]
    y_oh = batch["y_oh"]
    y_idx = batch["y_idx"]

    L_bce = F.binary_cross_entropy_with_logits(logits_sig, y_oh, pos_weight=pos_weight)
    L_ce  = F.cross_entropy(logits_sm, y_idx)

    p = torch.sigmoid(logits_sig)
    L_excl = ((p.sum(dim=1) - 1.0) ** 2).mean()

    L_monet = torch.tensor(0.0, device=logits_sig.device)
    if out["monet_hat"] is not None and batch["monet"].numel() > 0:
        L_monet = F.mse_loss(out["monet_hat"], batch["monet"])

    L_contrast = info_nce(out["pc"], out["pd"]) if cfg.w_contrast > 0 else torch.tensor(0.0, device=logits_sig.device)

    total = L_bce + cfg.w_ce*L_ce + cfg.w_excl*L_excl + cfg.w_monet*L_monet + cfg.w_contrast*L_contrast
    return total, {"bce": L_bce.item(), "ce": L_ce.item(), "excl": L_excl.item(),
                   "monet": float(L_monet.item()), "contrast": float(L_contrast.item())}


## 17) Freezing strategy + optimizer + scheduler

In [ ]:

def set_trainable_stage1(model: nn.Module):
    for p in model.parameters():
        p.requires_grad_(False)

    for m in model.modules():
        if isinstance(m, DualLoRALinear):
            m.A.requires_grad_(True)
            m.B.requires_grad_(True)

    for sub in [model.meta, model.fuser, model.head_sig, model.head_sm, model.proj]:
        for p in sub.parameters():
            p.requires_grad_(True)

    if model.head_monet is not None:
        for p in model.head_monet.parameters():
            p.requires_grad_(True)

def set_trainable_stage2(model: nn.Module, unfreeze_last_n_blocks: int = 2):
    set_trainable_stage1(model)
    bb = model.backbone.backbone
    blocks = getattr(bb, "blocks", None)
    if blocks is not None and unfreeze_last_n_blocks > 0:
        for blk in list(blocks)[-unfreeze_last_n_blocks:]:
            for p in blk.parameters():
                p.requires_grad_(True)

def build_optimizer(model: nn.Module, lr_heads: float, lr_lora: float, lr_unfreeze: float):
    lora_params, head_params, unfreeze_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if ".A" in name or ".B" in name:
            lora_params.append(p)
        elif "backbone.backbone.blocks" in name:
            unfreeze_params.append(p)
        else:
            head_params.append(p)

    groups = []
    if lora_params:     groups.append({"params": lora_params, "lr": lr_lora})
    if head_params:     groups.append({"params": head_params, "lr": lr_heads})
    if unfreeze_params: groups.append({"params": unfreeze_params, "lr": lr_unfreeze})

    opt = torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)
    for g in opt.param_groups:
        g["initial_lr"] = g["lr"]
    return opt

def cosine_lr(step, total_steps, base_lr, warmup_steps=0):
    if step < warmup_steps:
        return base_lr * (step / max(1, warmup_steps))
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


## 18) Train / validate loops

In [ ]:

@torch.no_grad()
def validate(model: nn.Module, loader: DataLoader):
    model.eval()
    all_probs, all_true = [], []
    for batch in loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
        batch["cat"] = {k: v.to(device) for k,v in batch["cat"].items()}

        out = model(batch)
        probs = torch.sigmoid(out["logits_sig"]).detach().cpu().numpy()
        all_probs.append(probs)
        all_true.append(batch["y_oh"].detach().cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    true  = np.concatenate(all_true, axis=0)
    return {"macro_f1@0.5": macro_f1_threshold05(probs, true)}

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, epoch: int, total_steps: int, warmup_steps: int):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running = 0.0
    step0 = epoch * len(loader)

    pbar = tqdm(enumerate(loader), total=len(loader))
    for step, batch in pbar:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
        batch["cat"] = {k: v.to(device) for k,v in batch["cat"].items()}

        for g in optimizer.param_groups:
            g["lr"] = cosine_lr(step0 + step, total_steps, g["initial_lr"], warmup_steps)

        with torch.cuda.amp.autocast(enabled=(cfg.use_amp and device.type == "cuda")):
            out = model(batch)
            loss, ld = compute_losses(out, batch)
            loss = loss / cfg.grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running += loss.item() * cfg.grad_accum
        pbar.set_postfix({"loss": running/(step+1), **{k: round(v,4) for k,v in ld.items()}})

    return {"train_loss": running/len(loader)}

def run_training():
    best = -1.0
    best_path = "best_milk10k_vit_latentfusion.pt"

    # Stage 1
    set_trainable_stage1(model)
    opt = build_optimizer(model, cfg.lr_heads, cfg.lr_lora, cfg.lr_unfreeze)
    total_steps = cfg.epochs_stage1 * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    for ep in range(cfg.epochs_stage1):
        tr = train_one_epoch(model, train_loader, opt, ep, total_steps, warmup_steps)
        va = validate(model, val_loader)
        print(f"[Stage1][{ep+1}/{cfg.epochs_stage1}] train={tr} val={va}")
        if va["macro_f1@0.5"] > best:
            best = va["macro_f1@0.5"]
            torch.save(model.state_dict(), best_path)
            print("Saved best:", best_path, "score:", best)

    # Stage 2
    model.load_state_dict(torch.load(best_path, map_location=device))
    set_trainable_stage2(model, unfreeze_last_n_blocks=2)
    opt = build_optimizer(model, cfg.lr_heads*0.33, cfg.lr_lora*0.5, cfg.lr_unfreeze)
    total_steps = cfg.epochs_stage2 * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    for ep in range(cfg.epochs_stage2):
        tr = train_one_epoch(model, train_loader, opt, ep, total_steps, warmup_steps)
        va = validate(model, val_loader)
        print(f"[Stage2][{ep+1}/{cfg.epochs_stage2}] train={tr} val={va}")
        if va["macro_f1@0.5"] > best:
            best = va["macro_f1@0.5"]
            torch.save(model.state_dict(), best_path)
            print("Saved best:", best_path, "score:", best)

    print("Best Macro-F1@0.5:", best, "checkpoint:", best_path)

# run_training()


## 19) Debug cell (one batch forward + shapes + loss)

In [ ]:

batch = next(iter(train_loader))
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
batch["cat"] = {k: v.to(device) for k,v in batch["cat"].items()}

with torch.no_grad():
    out = model(batch)

print("img_clin:", batch["img_clin"].shape)
print("img_derm:", batch["img_derm"].shape)
print("logits_sig:", out["logits_sig"].shape)
print("logits_sm:", out["logits_sm"].shape)
print("monet_hat:", None if out["monet_hat"] is None else out["monet_hat"].shape)

loss, ld = compute_losses(out, batch)
print("loss:", float(loss.item()), "parts:", ld)


## 20) Submission CSV generator (lesion + 11 probability columns)

In [ ]:

@torch.no_grad()
def predict_probs(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch in tqdm(loader, total=len(loader)):
        lesion_ids = batch["lesion"]
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
        batch["cat"] = {k: v.to(device) for k,v in batch["cat"].items()}

        out = model(batch)
        probs = torch.sigmoid(out["logits_sig"]).detach().cpu().numpy()
        for lid, p in zip(lesion_ids, probs):
            row = {"lesion": lid}
            for j, c in enumerate(cfg.classes):
                row[c] = float(p[j])
            rows.append(row)
    return pd.DataFrame(rows)

# Example:
# sub_df = predict_probs(model, val_loader)
# sub_df.to_csv("submission.csv", index=False)
# display(sub_df.head())
